In [1]:
%pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.1 MB/s eta 0:00:00ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 72.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import mlflow
import dagshub

dagshub.init(repo_owner='amama22', 
             repo_name='ieee-cis-fraud-detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=f5afe6f6-c586-4c76-bb1e-d2b6cc217d4e&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=54c9094c9ef10073d65f008ddfea984b3882eb919c92cfc57760bf8ac89d7a7b




Accessing as amama22

Initialized MLflow to track repo "amama22/ieee-cis-fraud-detection"

Repository amama22/ieee-cis-fraud-detection initialized!

In [3]:
model_uri = "models:/LightGBM_Fraud_Detection_Best/2"
pipeline = mlflow.sklearn.load_model(model_uri)

In [8]:
def reduce_memory(df):
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    return df

test = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')
test_id.columns = [c.replace('-', '_') for c in test_id.columns]
test = reduce_memory(test.merge(test_id, on='TransactionID', how='left'))
transaction_ids = test['TransactionID']
X_test = test.drop(columns=['TransactionID'])

In [9]:
predictions = pipeline.predict_proba(X_test)[:, 1]
print("Predictions shape:", predictions.shape)
print("Prediction range:", predictions.min(), "-", predictions.max())
print("Mean prediction:", predictions.mean())

Predictions shape: (506691,)
Prediction range: 2.3425385997935116e-05 - 0.9998310752484693
Mean prediction: 0.15960759521000892


In [10]:
submission = pd.DataFrame({
    'TransactionID': transaction_ids,
    'isFraud': predictions
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10))
print("\nSubmission shape:", submission.shape)

   TransactionID   isFraud
0        3663549  0.028261
1        3663550  0.023642
2        3663551  0.053456
3        3663552  0.033716
4        3663553  0.020385
5        3663554  0.088584
6        3663555  0.112809
7        3663556  0.090744
8        3663557  0.001423
9        3663558  0.101226

Submission shape: (506691, 2)
